# Retrieval Augmented Generation : Pipeline 1 is Data Ingestion & Pipeline 2 is Augmented Generation

# Langchain Document Structure
```
doc = Document(
    pageContent: str,
    metadata: {dict}
)

```

```
from langchain_core.documents import Document 

doc = Document(
    page_content = "main text content",
    metadata = {
        "source":"sample.txt",
        "pages": 100,
        "author":"Rudyard Kipling"
    }
) 

```
Documentation Page Link: https://docs.langchain.com/oss/javascript/integrations/document_loaders/index#interface

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import re
import os 


c:\Users\user\OneDrive\Desktop\lex-connect-v2\python-services\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_pdfs(pdf_directory)-> list:
    """Loading all PDF's"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF's ")
    print("Loading...")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"✅ Loaded {len(all_documents)}")


        except Exception as e:
            print("❌ Couldn't load documents")
            print(f"Error {e}")

    
    return all_documents



all_docs = load_pdfs("./kaggle_dataset")
print(all_docs)

Found 0 PDF's 
Loading...
[]


# Common Questions to ask:
- “What should an NDA contain?”
- “How does a lease agreement work?”
- “Generate a separation agreement”
- “What documents are needed for adoption deed?”

In [3]:
# READ THE .MD FILE TO UNDERSTAND CHUNKING
def split_documents(documents):
    """Split the documents
    in such a way that we preserve semantic meaning
    across Clauses, Sections etc. etc.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = 2200,
        chunk_overlap = 300,
        length_function = len,
        separators = [
            "\n\nSECTION ",
            "\n\nARTICLE ",
            "\n\nCHAPTER ",
            "\n\n",
            ". ",
            "; ",
            " "
        ]
    )

    split_docs = []

    for doc in documents:

        text = doc.page_content

        # Before anything, create segments for major legal sections.
        sections = re.split(
            r'(?=\n(?:SECTION|Section|ARTICLE|Article|CHAPTER|Chapter))',
            text
        )


        for section in sections:
            chunks = splitter.create_documents(
                [section],
                metadatas = [doc.metadata]
            )


            split_docs.extend(chunks)

    return split_docs 


split_docs = split_documents(all_docs)
for chunk in split_docs:
    print(chunk)
    